In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.sans-serif'] = "Arial"
# Then, "ALWAYS use sans-serif fonts"
matplotlib.rcParams['font.family'] = "sans-serif"
from adjustText import adjust_text
matplotlib.rcParams['font.sans-serif'] = "Arial"
# Then, "ALWAYS use sans-serif fonts"
matplotlib.rcParams['font.family'] = "sans-serif"
from adjustText import adjust_text
import gseapy as gp 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib import rcParams
from scipy import stats
from gseapy import barplot, dotplot
import os
from gseapy.plot import gseaplot2

In [ ]:
df_2m = pd.read_csv('../new_mageck/Vglut2-GW-all_2M/Vglut2-GW-all_2M.gene_summary.txt', sep='\t')
hits_2m = df_2m[df_2m['hits'] == -1]['human homolog'].tolist()
df_4m = pd.read_csv('../new_mageck/Vglut2-GW-all_4M/Vglut2-GW-all_4M.gene_summary.txt', sep='\t')
hits_4m = df_4m[df_4m['hits'] == -1]['human homolog'].tolist()
df_18m = pd.read_csv('../new_mageck/Vglut2-GW-all_18M/Vglut2-GW-all_18M.gene_summary.txt', sep='\t')
hits_18m = df_18m[df_18m['hits'] == -1]['human homolog'].tolist()


In [ ]:
aging_specific = set(hits_18m) - set(hits_4m+hits_2m) 

In [ ]:
# Step 1: Define your gene list  
gene_list = list(aging_specific)  # Example gene list  

background_genes = df_18m['human homolog'].astype(str).tolist()  # Example background list  

# Step 2: Perform the KEGG enrichment analysis  
enrichment = gp.enrichr(  
gene_list=gene_list,  
gene_sets=['DisGeNET','KEGG_2021_Human', 'GTEx_Aging_Signatures_2021', 
          'GO_Biological_Process_2025', 'GO_Cellular_Component_2025','Aging_Perturbations_from_GEO_up',
          'Aging_Perturbations_from_GEO_down','ClinVar_2025', 'GTEx_Tissues_V8_2023'],
        background=background_genes, no_plot = True)  

enrichment_sig = enrichment.results[enrichment.results['Adjusted P-value']<0.05]
enrichment_sig.to_csv('../aging/18M_specific_hits_enrichment_sig.csv')


In [ ]:
rcParams['font.family'] = 'sans-serif'  
rcParams['font.sans-serif'] = ['Arial']  
rcParams['font.size'] = 15
rcParams['axes.linewidth'] = 0.8  
rcParams['xtick.major.width'] = 0.8  
rcParams['ytick.major.width'] = 0.8  
rcParams['xtick.major.size'] = 3  
rcParams['ytick.major.size'] = 3  
rcParams['pdf.fonttype'] = 42  # Ensures text is editable in AI  
rcParams['ps.fonttype'] = 42  
gene_set = 'GO_Cellular_Component_2025'
temp = enrichment_sig[enrichment_sig['Gene_set'] == gene_set]
top_kegg = temp.head(8)
top_kegg['-log10 P'] = -np.log10(top_kegg['Adjusted P-value'])
top_kegg['Term'] = top_kegg['Term'].apply(lambda x: x.split(' (GO')[0])
# top_kegg['Term'] = top_kegg['Term'].apply(lambda x: x[:30] + '...' if len(x) > 30 else x)  
plt.figure(figsize=(8, 6))  
palette = 'viridis_r'

sns.barplot(x='-log10 P', y='Term', data=top_kegg, palette=palette)  
plt.title(f'Significant {gene_set} Enrichments')  
plt.xlabel('-log10 Adjusted P-value')  
plt.ylabel(gene_set)  
plt.tight_layout()  
plt.savefig(f'../aging/18m_specific_hits_enrichment_GOCC.pdf', bbox_inches = 'tight') 

In [ ]:
df_aging = pd.read_csv('../related_files/human_brain_aging_DEG.csv')

df = df_aging[df_aging['cell type'] == 'ext']

In [ ]:
df_overlap = df[df['gene name'].isin(aging_specific)]

In [ ]:
# Step 1: Define your gene list  
gene_list = df_overlap['gene name'].tolist()  # Example gene list  

# background_genes = df_18m['human homolog'].astype(str).tolist()  # Example background list  

# Step 2: Perform the KEGG enrichment analysis  
enrichment = gp.enrichr(  
gene_list=gene_list,  
gene_sets=['KEGG_2021_Human','GO_Biological_Process_2025', 'GO_Cellular_Component_2025'],
        background=background_genes, no_plot = True)  

enrichment_sig = enrichment.results[enrichment.results['Adjusted P-value']<0.05]
enrichment_sig.to_csv('../aging/18M_specific_hits_single_cell_overlap_enrichment_sig.csv')


In [ ]:
group_colors.items()

In [ ]:
group_colors = {
    "ETC/OXPHOS": "#702F8A",       # dark violet
    "Mito translation": "#1F7A1F", # deep green
    "Cyto translation": "#CC5500", # dark orange
    "Mito import": "#B03060",      # deep rose/magenta
    "Others": "#666666"            # neutral medium gray
}
# -------------------------
# 6) Plot with gene labels (Others in background)
# -------------------------
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 13,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False
})

fig, ax = plt.subplots(figsize=(8, 4), dpi=300)

# Background threshold spans
y_thr = -np.log10(q_thr)
ax.axvspan(-lfc_thr, lfc_thr, color="#bfbfbf", alpha=0.18, zorder=0)
ax.axhspan(0, y_thr, color="#bfbfbf", alpha=0.18, zorder=0)

# Context (all points, faint)
ax.scatter(
    df_ext[col_lfc], df_ext["neglog10_q"],
    s=10, color="black", alpha=0.04, linewidths=0, zorder=1
)

# Up/Down significant base layers
ax.scatter(
    df_ext.loc[is_up, col_lfc], df_ext.loc[is_up, "neglog10_q"],
    s=10, color=col_up, edgecolor="white", linewidth=0.4, alpha=1, zorder=2,
    label="Up"
)
ax.scatter(
    df_ext.loc[is_down, col_lfc], df_ext.loc[is_down, "neglog10_q"],
    s=10, color=col_down, edgecolor="white", linewidth=0.4, alpha=1, zorder=2,
    label="Down"
)

# --- Step 1: plot "Others" first (behind everything else) ---
others_sub = df_hl_all[df_hl_all["Group"] == "Others"]
if not others_sub.empty:
    ax.scatter(
        others_sub[col_lfc],
        others_sub["neglog10_q"],
        s=25, c=group_colors["Others"],
        edgecolor="white", linewidth=0.5, alpha=1, zorder=3,
        label="Others"
    )

# --- Step 2: plot the 4 main groups on top ---
for group_name in ["ETC/OXPHOS", "Mito translation", "Cyto translation", "Mito import"]:
    if group_name not in df_hl_all["Group"].unique():
        continue
    color = group_colors[group_name]
    sub = df_hl_all[df_hl_all["Group"] == group_name]
    
    ax.scatter(
        sub[col_lfc], sub["neglog10_q"],
        s=30, c=color, edgecolor="white", linewidth=0.6,
        alpha=0.9, zorder=5, label=group_name
    )
    
#     # Annotate genes
#     for _, row in sub.iterrows():
#         ax.text(
#             row[col_lfc] + 0.05,
#             row["neglog10_q"] + 5,
#             row[col_gene],
#             fontsize=8, color=color,
#             ha="left", va="bottom", zorder=6
#         )

# Axes lines and thresholds
ax.set_xlabel("log2 Fold Change (Elderly vs Adult)")
ax.set_ylabel("-log10(q-value)")
ax.axvline( lfc_thr,  color="0.4", lw=1.0, ls="--", alpha=0.8, zorder=4)
ax.axvline(-lfc_thr,  color="0.4", lw=1.0, ls="--", alpha=0.8, zorder=4)
ax.axhline( y_thr,    color="0.4", lw=1.0, ls="--", alpha=0.8, zorder=4)

# Limits
max_abs_lfc = np.nanmax(np.abs(df_ext[col_lfc].values))
pad = 0.06 * max_abs_lfc
ax.set_xlim(-(max_abs_lfc + pad), (max_abs_lfc + pad))
y_top = np.clip(df_ext["neglog10_q"].quantile(0.995), 2, None)
ax.set_ylim(0, y_top + 20)

# --- Legend: ensure 'Others' is last ---
handles, labels = ax.get_legend_handles_labels()
uniq = dict(zip(labels, handles))

# Move 'Others' to the end if present
if "Others" in uniq:
    ordered_labels = [l for l in uniq.keys() if l != "Others"] + ["Others"]
else:
    ordered_labels = list(uniq.keys())

ordered_handles = [uniq[l] for l in ordered_labels]

ax.legend(ordered_handles, ordered_labels, frameon=False, loc=0)

plt.tight_layout()
plt.savefig("volcano_ext_labeled_groups_others_back_nogenelabel.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
df_top = df_hl_all[df_hl_all['neglog10_q'] ==300]
for i in df_top[df_top['Group'] != 'Others'].sort_values('log2 fold change of elderly vs adult')['gene name']:
    print(i)

In [ ]:
group_colors = {
    "ETC/OXPHOS": "#702F8A",       # dark violet
    "Mito Translation": "#1F7A1F", # deep green
    "Ribosome Biogenesis": "#CC5500", # dark orange
}
gene_groups = {
    'Mito Translation':[
    "MRPS17", "GFM1", "MRPS15", "MRPS16", "GFM2", "MRPS14", "MRPS11", "MRPS33", "MRPS34",
    "MRPL38", "MRPL16", "MRPS31", "MRPL39", "MRPL15", "MRPL12", "MRPL13", "MRPL10",
    "MRPL41", "MRPL42", "MRPL9", "EARS2", "RARS2", "MRPS26", "MRPS23", "GATC", "MRPS2",
    "MRPL45", "QRSL1", "MRPL46", "MRPL24", "MRPS5", "MRPL21", "MRPL44", "MRPS18C",
    "LARS2", "MRPS9", "MRPL51"
], "ETC/OXPHOS":[
    "NDUFB8", "NDUFB7", "NDUFB10", "NDUFB6", "UQCRB", "NDUFA3", "NDUFB2", "NDUFA1",
    "NDUFC2", "SDHC", "COX5A", "COX6B1", "NDUFS3", "UQCRC1", "UQCRFS1", "NDUFAF1",
    "NDUFA11", "ATP5MF", "ATP5ME"
], "Ribosome Biogenesis":[
    "DDX27", "PAK1IP1", "HEATR1", "FCF1", "NOC4L", "RRP9", "MRM3", "NOL6", "RRP7A",
     "RPS14", "EMG1", "XPO1", "EXOSC4", "NOB1", "BMS1", "DHX37", "UTP20",
    "RIOK2", "RIOK1", "EXOSC3", "EXOSC2", "NOM1", "NOP14", "NOP58", "UTP6", "RPS7",
    "KRR1", "NGDN", "DDX10", "WDR75", "BYSL", "BOP1", "EBNA1BP2", "TBL3", "MPHOSPH10"
]
}

# 4M vs. 18M


df = pd.read_csv('../new_mageck/Vglut2-GW-all_2M/Vglut2-GW-all_2M.gene_summary.txt', sep='\t')
df['signed_score'] = df.apply(lambda x: np.log10(x['neg|score']) if x['neg|lfc']<0 else -np.log10(x['pos|score']), axis=1)
screen1_data = df



df = pd.read_csv('../new_mageck/Vglut2-GW-all_18M/Vglut2-GW-all_18M.gene_summary.txt', sep='\t')
df['signed_score'] = df.apply(lambda x: np.log10(x['neg|score']) if x['neg|lfc']<0 else -np.log10(x['pos|score']), axis=1)
screen2_data = df


# Merge datasets for comparison  
merged_data = pd.merge(  
    screen1_data,   
    screen2_data,   
    on = 'id',
    how='inner',   
    suffixes=('_screen1', '_screen2')  
)  


print(f"Merged data: {len(merged_data)} genes")  
def categorize_hits(row):  
    sig1 = (row['human homolog_screen2'] in gene_groups['Mito Translation'])
    sig2 = (row['human homolog_screen2'] in gene_groups['ETC/OXPHOS'])
    sig3 = (row['human homolog_screen2'] in gene_groups['Ribosome Biogenesis'])
    if sig1:  
        return "Mito Translation"  
    elif sig2:  
        return "ETC/OXPHOS"  
    elif sig3:  
        return "Ribosome Biogenesis"  
    else:  
        return "Others" 


merged_data['hit_category'] = merged_data.apply(categorize_hits, axis=1)  

# Set up Cell-style colors - clean, distinctive but not overwhelming  
category_colors = {  
"ETC/OXPHOS": "#702F8A",       # dark violet
    "Mito Translation": "#1F7A1F", # deep green
    "Ribosome Biogenesis": "#CC5500", # dark orange 
    "Others": "#DDDDDD"   # Light gray  
}  

# Text color - Cell papers typically use dark gray instead of pure black  
cell_text_color = "#333333"  
rcParams['font.size'] = 8

# 1. Create LFC comparison scatter plot with Cell-style aesthetics  
fig, ax = plt.figure(figsize=(3, 3), dpi=300), plt.gca()  

# First plot all non-significant points (background)  
nonsig_data = merged_data[merged_data['hit_category'] == "Others"]  
ax.scatter(  
    nonsig_data['signed_score_screen1'],   
    nonsig_data['signed_score_screen2'],  
    c=category_colors["Others"],  
    alpha=0.8,  
    s=5,  
    linewidths=0,  
    rasterized=True  # Better for large datasets in PDF  
)  

# Then plot significant hits on top  
for category in ["Mito Translation", "ETC/OXPHOS", "Ribosome Biogenesis"]:  
    cat_data = merged_data[merged_data['hit_category'] == category]  
    ax.scatter(  
        cat_data['signed_score_screen1'],   
        cat_data['signed_score_screen2'],  
        c=category_colors[category],  
        alpha=1,  
        s=10,  
        linewidths=0.2,  
        edgecolors='white'  
    )  
from adjustText import adjust_text

texts = []  # store labels for adjustment


# )
# Set limits with some padding  
xlim = ax.get_xlim()  
ylim = ax.get_ylim()  
min_lim = min(xlim[0], ylim[0])  
max_lim = max(xlim[1], ylim[1])  
padding = 0.
ax.set_xlim(min_lim - padding, max_lim + padding)  
ax.set_ylim(min_lim - padding, max_lim + padding)  

# Add a diagonal reference line  
ax.plot([min_lim, max_lim], [min_lim, max_lim], color='#888888', linestyle='--', alpha=0.8, linewidth=0.8) 
ax.plot([min_lim, max_lim], [0, 0], color='#888888', linestyle='-', alpha=0.8, linewidth=0.8) 
ax.plot([0, 0], [min_lim, max_lim], color='#888888', linestyle='-', alpha=0.8, linewidth=0.8) 
screen = 'Vglut2'
# Style the axis labels  
ax.set_xlabel(f'{screen} 4M', fontsize=8)  
ax.set_ylabel(f'{screen} 18M', fontsize=8)  
ax.set_title(f'{screen} 4M vs. 18M', fontsize=8)  
# --- Add a Cell-style legend for the three functional categories ---
import matplotlib.patches as mpatches

legend_elements = [
    mpatches.Patch(color=category_colors["Mito Translation"],  label="Mito Translation"),
    mpatches.Patch(color=category_colors["ETC/OXPHOS"],        label="ETC/OXPHOS"),
    mpatches.Patch(color=category_colors["Ribosome Biogenesis"], label="Ribosome Biogenesis"),
    mpatches.Patch(color=category_colors["Others"],             label="Others")
]

# Position legend cleanly without overlapping data
ax.legend(
    handles=legend_elements,
    loc='upper left',           # position can be 'upper right' depending on data spread
    frameon=False,              # no box outline
    fontsize=8)

plt.savefig('../aging/4M_vs_18M_cat.png', dpi=600, bbox_inches='tight')  
